# FDTD シミュレーション実行ノートブック（矩形 Kukei）

`UT_FlawVariation_Sim.run_simulation()` を **1条件** でも **複数条件** でも回せるようにしたもの。

| 使い方 | セル |
|---|---|
| 準備（毎回最初に実行） | 1〜3 |
| 1条件だけ回す | 4〜6 |
| 複数条件を回す | 7〜10 |

**所要時間の目安: 1条件あたり約 2.5 分**（19,586 ステップ × 約 7 ms + 初期化 約 9 秒）。
複数条件を回す前に必ず「実行計画」セルで総時間を確認すること。

**前提**: Anaconda の `venv` 環境（CuPy + CUDA）。macOS では CuPy が無いので実行できない。


## 1. 準備

In [ ]:
import os
import sys
import time
import itertools
from pathlib import Path

# run_simulation() は出力先を "tmp_output/sim" という**相対パス**で持っている。
# ノートブックの CWD は既定で .ipynb のある場所（vscode/simulation/）なので、
# そのまま実行すると vscode/simulation/tmp_output/sim に書かれてしまう。
# リポジトリルートに移動して、他のスクリプトと出力先を揃える。
_here = Path.cwd()
REPO_ROOT = _here
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "AGENTS.md").exists():
    raise RuntimeError(f"リポジトリルートが見つからない（起点: {_here}）")

os.chdir(REPO_ROOT)
SIM_DIR = REPO_ROOT / "vscode" / "simulation"
if str(SIM_DIR) not in sys.path:
    sys.path.insert(0, str(SIM_DIR))

print("リポジトリルート:", REPO_ROOT)
print("カレント        :", Path.cwd())

### GPU の確認

CuPy が CUDA を掴めているか。ここで落ちるなら `SETUP.md` の DLL 設定を確認する。

In [ ]:
try:
    import cupy as cp
    _dev = cp.cuda.Device()
    _props = cp.cuda.runtime.getDeviceProperties(_dev.id)
    _free, _total = _dev.mem_info
    print("CuPy      :", cp.__version__)
    print("GPU       :", _props["name"].decode())
    print("メモリ    : {:.1f} / {:.1f} GB 空き".format(_free / 1e9, _total / 1e9))
    # elementwise が実際に動くかまで確認する（DLL 不備はここで出る）
    print("動作確認  :", float((cp.arange(10, dtype=cp.float64) ** 2).sum()), "(285.0 なら正常)")
    GPU_OK = True
except Exception as e:
    GPU_OK = False
    print("CuPy が使えない:", type(e).__name__, e)
    print()
    print("  - macOS では CuPy 非対応のため、このノートブックは実行できない")
    print("  - Windows で出た場合は SETUP.md の CuPy DLL 設定を確認")

### ソルバと共通関数の読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import japanize_matplotlib  # noqa: F401  日本語ラベル用
except ImportError:
    pass

import UT_FlawVariation_Sim as sim
from naming import SIM_SHAPE, sim_csv_name
import sourse_new_2 as sn2   # dt, make_fftdata（他のプロット系と揃えるため）

# --- 出力先 ---
# run_simulation() の書き出し先。他の場所にも過去の出力があるので、
# 既存チェックは3か所すべてを見る（同じ条件を2.5分かけて回し直さないため）。
OUT_DIR = REPO_ROOT / "tmp_output" / "sim"
SEARCH_DIRS = [
    OUT_DIR,
    REPO_ROOT / "tmp_output" / "sim_csv",     # calculate_avg_waveforms.py が読む場所
    REPO_ROOT / "data" / "simdata_out",       # loop_UT_sim.py が見る場所
]
SEC_PER_RUN = 150   # 1条件の実測目安 [s]（CLAUDE.md）


def out_name(pitch_mm, width_mm, depth_mm):
    """命名規則は AGENTS.md 7章。sim_csv_name は [m] を受け取る。"""
    return sim_csv_name(SIM_SHAPE, pitch_mm / 1000, width_mm / 1000, depth_mm / 1000)


def find_existing(pitch_mm, width_mm, depth_mm):
    """既に計算済みならそのパスを返す。無ければ None。"""
    name = out_name(pitch_mm, width_mm, depth_mm)
    for d in SEARCH_DIRS:
        p = d / name
        if p.exists():
            return p
    return None


def run_one(pitch_mm, width_mm, depth_mm, overwrite=False):
    """1条件を実行して出力パスを返す。既存ならスキップ。

    注意: run_simulation() の引数順は (f_width, f_pitch, f_depth) で、
    ファイル名の pitch → width → depth 順とは**違う**。取り違え防止のため
    必ずキーワード引数で渡すこと。
    """
    got = find_existing(pitch_mm, width_mm, depth_mm)
    if got is not None and not overwrite:
        print(f"スキップ（計算済み）: {got.relative_to(REPO_ROOT)}")
        return got

    t0 = time.time()
    sim.run_simulation(
        f_width=width_mm / 1000,
        f_pitch=pitch_mm / 1000,
        f_depth=depth_mm / 1000,
    )
    print(f"  所要 {time.time() - t0:.1f} s")
    return OUT_DIR / out_name(pitch_mm, width_mm, depth_mm)


def load_wave(path):
    w = np.loadtxt(path, delimiter=",")
    return w[:, 0] if w.ndim > 1 else w


print("出力先:", OUT_DIR.relative_to(REPO_ROOT))
print("形状  :", SIM_SHAPE)
print("dt    : {:.4e} s  → 19586 ステップ = {:.2f} us".format(sn2.dt, 19586 * sn2.dt * 1e6))

---
## 2. 1条件だけ回す

**ここを編集する。** 単位は **mm**（`run_simulation` は [m] だが、このノートブックは
命名規則（AGENTS.md 7章）に合わせて mm で受け取り、内部で変換する）。

In [ ]:
# ===== ここを編集 =====
PITCH_MM = 2.00    # ピッチ [mm]
WIDTH_MM = 0.15    # 溝の幅 [mm]
DEPTH_MM = 0.20    # 溝の深さ [mm]

OVERWRITE = False  # True にすると計算済みでも上書きして回し直す
# ======================

print("条件  : pitch {} / width {} / depth {} [mm]".format(PITCH_MM, WIDTH_MM, DEPTH_MM))
print("出力名:", out_name(PITCH_MM, WIDTH_MM, DEPTH_MM))
_done = find_existing(PITCH_MM, WIDTH_MM, DEPTH_MM)
print("状態  :", f"計算済み（{_done.parent.name}/）" if _done else "未計算 → 約 2.5 分")

実行（約 2.5 分）。

In [ ]:
path = run_one(PITCH_MM, WIDTH_MM, DEPTH_MM, overwrite=OVERWRITE)
wave = load_wave(path)
print("波形長:", len(wave), " ({:.2f} us)".format(len(wave) * sn2.dt * 1e6))

### 結果の確認（時間波形と FFT スペクトル）

In [ ]:
t_us = np.arange(len(wave)) * sn2.dt * 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(t_us, wave, lw=0.8)
axes[0].set_xlabel("時間 [us]")
axes[0].set_ylabel("T1（探触子開口平均）")
axes[0].set_title("時間波形  pitch{} width{} depth{} [mm]".format(PITCH_MM, WIDTH_MM, DEPTH_MM))
axes[0].grid(alpha=0.3)

# 底面往復の目安（縦波・板厚 20 mm）。どのエコーを見ているかの当たりを付ける用
_rt_us = 2 * sn2.x_length / sn2.cl * 1e6
for _k in (1, 2):
    if _k * _rt_us < t_us[-1]:
        axes[0].axvline(_k * _rt_us, color="gray", ls=":", lw=1)
        axes[0].text(_k * _rt_us, axes[0].get_ylim()[1], f" {_k}次", va="top", fontsize=9, color="gray")

# 初期パルスを避けて切り出してから FFT（extract_features.py と同じ扱い）
_seg = sn2.kiritori2(wave[8000:], sn2.left, sn2.right)
mag, freq = sn2.make_fftdata(_seg, sn2.dt)
_band = (freq >= 0) & (freq <= 15e6)
axes[1].plot(freq[_band] / 1e6, mag[_band], lw=1.0)
axes[1].axvline(4.7, color="crimson", ls="--", lw=1, label="送信 4.7 MHz")
axes[1].set_xlabel("周波数 [MHz]")
axes[1].set_ylabel("振幅")
axes[1].set_title("FFT スペクトル")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

---
## 3. 複数条件を回す

**ここを編集する。** 2 通りの指定方法がある。

- `USE_SIMDATA_CSV = False`: 下のリストの**全組み合わせ**（`itertools.product`）
- `USE_SIMDATA_CSV = True` : `vscode/simulation/simdata.csv` から読む（`loop_UT_sim.py` と同じ）

計算済みの条件は自動でスキップするので、**途中で中断しても同じセルを再実行すれば続きから**進む。

In [ ]:
# ===== ここを編集 =====
USE_SIMDATA_CSV = False

# USE_SIMDATA_CSV = False のとき使う（単位 mm）
PITCH_LIST = [1.25, 2.00]
WIDTH_LIST = [0.15, 0.40, 0.60, 0.80]
DEPTH_LIST = [0.10, 0.15, 0.20]

SKIP_EXISTING = True   # False にすると計算済みでも全部回し直す
# ======================

if USE_SIMDATA_CSV:
    from loop_UT_sim import load_simdata
    _p = load_simdata(str(SIM_DIR / "simdata.csv"))
    # simdata.csv は [m] 単位なので mm に直す
    combos = [(p * 1000, w * 1000, d * 1000)
              for p, w, d in itertools.product(_p["f_pitch"], _p["f_width"], _p["f_depth"])]
    print("simdata.csv から", len(combos), "条件")
else:
    combos = list(itertools.product(PITCH_LIST, WIDTH_LIST, DEPTH_LIST))
    print("リストの直積:", len(combos), "条件")

### 実行計画の確認

**実行前に必ずこのセルで総時間を見ること。**

In [ ]:
todo, done = [], []
for c in combos:
    (done if (SKIP_EXISTING and find_existing(*c)) else todo).append(c)

print("全条件   : {}".format(len(combos)))
print("計算済み : {}（スキップ）".format(len(done)))
print("これから : {}".format(len(todo)))
_est = len(todo) * SEC_PER_RUN
print("推定時間 : {:.0f} 分（{:.1f} 時間）".format(_est / 60, _est / 3600))
print()
for i, (p, w, d) in enumerate(todo[:30], 1):
    print("  [{:>3}] pitch {:<5} width {:<5} depth {:<5} -> {}".format(
        i, p, w, d, out_name(p, w, d)))
if len(todo) > 30:
    print("  ... 他 {} 条件".format(len(todo) - 30))

### 実行

上の推定時間を確認してから実行する。中断しても再実行で続きから進む。

In [ ]:
results = []
t_start = time.time()

for i, (p, w, d) in enumerate(todo, 1):
    elapsed = time.time() - t_start
    eta = (len(todo) - i + 1) * (elapsed / max(i - 1, 1)) if i > 1 else len(todo) * SEC_PER_RUN
    print("\n[{}/{}] pitch {} / width {} / depth {} [mm]  "
          "（経過 {:.1f} 分, 残り 約 {:.1f} 分）".format(i, len(todo), p, w, d, elapsed / 60, eta / 60))
    try:
        results.append((p, w, d, run_one(p, w, d, overwrite=not SKIP_EXISTING)))
    except Exception as e:
        # 1条件こけても残りは続ける（2.5分×N を無駄にしないため）
        print("  失敗:", type(e).__name__, e)

print("\n完了: {} 条件, 合計 {:.1f} 分".format(len(results), (time.time() - t_start) / 60))

### 結果の一覧

計算済みも含めて、指定した全条件の波形を重ねて表示する。

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
n_plotted = 0

for p, w, d in combos:
    path = find_existing(p, w, d)
    if path is None:
        continue
    v = load_wave(path)
    tt = np.arange(len(v)) * sn2.dt * 1e6
    # 振幅で正規化して重ねる（条件間の形の違いを見るため）
    ax.plot(tt, v / np.abs(v).max(), lw=0.7, alpha=0.75,
            label="p{:g} w{:g} d{:g}".format(p, w, d))
    n_plotted += 1

ax.set_xlabel("時間 [us]")
ax.set_ylabel("T1（最大値で正規化）")
ax.set_title("{} 条件の比較".format(n_plotted))
ax.grid(alpha=0.3)
if 0 < n_plotted <= 12:
    ax.legend(ncol=3, fontsize=9)
fig.tight_layout()
plt.show()

print("表示: {} / {} 条件（残りは未計算）".format(n_plotted, len(combos)))

---
## 4. 物質中の応力変化アニメーション

`run_simulation()` に `snapshot_every` を渡すと、**波形 CSV を作るのと同じ実行**から
応力場のスナップショットも保存する（NPZ、`tmp_output/field/`）。
別実装の `kukei_ani.py` ではなく正のソルバを使うので、**アニメーションと波形が厳密に対応する**。

表示は2通り切り替えられる。

| モード | パネル | 用途 |
|---|---|---|
| `"PS"` | `(T1+T3)/2` と `T5` | **P波とS波を分けて見る**（モード変換用） |
| `"T"` | `T1` / `T3` / `T5` | 応力テンソルの3成分をそのまま |

`(T1+T3)/2` は体積変化、`T5`(=σxz) はせん断に対応するので、前者が P 波、後者が S 波の
目安になる。**完全な分離ではない**（斜め入射の P 波も σxz を持つ）が、欠陥の角で P が S に
変わる様子はこの対比で追える。

**データ量**（3成分・全体図のみ・19,586 ステップ）

| 間引き | 保存間隔 | フレーム | 容量 |
|---|---|---|---|
| 1/8 | 100 | 195 | 約 294 MB |
| 1/8 | 200 | 97 | 約 147 MB |
| 1/12 | 100 | 195 | 約 131 MB |


### 設定

In [ ]:
# ===== ここを編集 =====
ANI_PITCH_MM = 2.00
ANI_WIDTH_MM = 0.15
ANI_DEPTH_MM = 0.20

SNAPSHOT_EVERY = 100        # 何ステップごとに場を保存するか
SNAPSHOT_DOWNSAMPLE = 8     # 全体図の間引き率
SNAPSHOT_ZOOM_MM = (3.0, 0.5)   # 底面中央の拡大領域 (幅mm, 高さmm)。None で拡大なし
# ======================

_n_frames = 19586 // SNAPSHOT_EVERY
_mb = _n_frames * (32.0 / SNAPSHOT_DOWNSAMPLE ** 2) * 3
print("フレーム数: {}（{:.3f} us 間隔）".format(_n_frames, SNAPSHOT_EVERY * sn2.dt * 1e6))
print("推定容量  : 約 {:.0f} MB（圧縮前・3成分）".format(_mb))

from naming import ani_npz_name
NPZ_PATH = REPO_ROOT / "tmp_output" / "field" / ani_npz_name(
    "kukei_field", sim.probe_d, ANI_PITCH_MM / 1000, ANI_WIDTH_MM / 1000, ANI_DEPTH_MM / 1000)
print("NPZ       :", NPZ_PATH.name)
print("状態      :", "取得済み" if NPZ_PATH.exists() else "未取得 → 次のセルで約 2.5 分")

### 実行（約 2.5 分）

既に NPZ があればスキップする。取り直したいときは `FORCE = True`。

In [ ]:
FORCE = False

if NPZ_PATH.exists() and not FORCE:
    print("スキップ（取得済み）:", NPZ_PATH.relative_to(REPO_ROOT))
else:
    _t0 = time.time()
    _res = sim.run_simulation(
        f_width=ANI_WIDTH_MM / 1000,
        f_pitch=ANI_PITCH_MM / 1000,
        f_depth=ANI_DEPTH_MM / 1000,
        snapshot_every=SNAPSHOT_EVERY,
        snapshot_downsample=SNAPSHOT_DOWNSAMPLE,
        snapshot_zoom_mm=SNAPSHOT_ZOOM_MM,
    )
    print("所要 {:.1f} 分".format((time.time() - _t0) / 60))
    NPZ_PATH = Path(_res["npz_path"])

### 読み込みと表示の準備

In [ ]:
from matplotlib import animation
from matplotlib.gridspec import GridSpec

FIELD = np.load(NPZ_PATH)
print("成分      :", [k for k in FIELD.files if k.startswith(("g_", "z_"))])
print("フレーム  :", FIELD["g_T1"].shape[0], "枚  格子", FIELD["g_T1"].shape[1:])
print("フレーム間隔: {:.4f} us".format(float(FIELD["frame_dt"]) * 1e6))
print("条件 (mm) : pitch {} width {} depth {}".format(*FIELD["params_mm"]))


def panels(mode="PS", zoom=False):
    """表示するパネルを (タイトル, 配列[frame, x, z]) のリストで返す。

    mode="PS": (T1+T3)/2 を P 波、T5 を S 波の目安として2枚
    mode="T" : T1 / T3 / T5 をそのまま3枚
    """
    pre = "z_" if zoom else "g_"
    if zoom and (pre + "T1") not in FIELD.files:
        raise KeyError("拡大領域が保存されていない（SNAPSHOT_ZOOM_MM=None で実行した）")
    if mode == "PS":
        return [("P 波  (T1+T3)/2", (FIELD[pre + "T1"] + FIELD[pre + "T3"]) / 2),
                ("S 波  T5 (sigma_xz)", FIELD[pre + "T5"])]
    if mode == "T":
        return [("T1 (sigma_xx)", FIELD[pre + "T1"]),
                ("T3 (sigma_zz)", FIELD[pre + "T3"]),
                ("T5 (sigma_xz)", FIELD[pre + "T5"])]
    raise ValueError('mode は "PS" か "T"')


def _extent(zoom):
    return list(FIELD["zoom_extent_mm"] if zoom else FIELD["extent_mm"])


def _scale(arr, saturate):
    """色スケール。全フレームの最大値の saturate 倍で頭打ちにする。

    時間とともに振幅が落ちるので、最大値そのままだと後半が真っ白になる。
    弱い散乱波（モード変換した成分）を見たいときは saturate を小さくする。
    """
    m = float(np.abs(arr).max())
    return max(m * saturate, 1e-30)


# 波形（同じ条件のもの）。アニメの時刻カーソルに使う
_wave_path = find_existing(ANI_PITCH_MM, ANI_WIDTH_MM, ANI_DEPTH_MM)
WAVE = load_wave(_wave_path) if _wave_path else None
print("波形      :", _wave_path.name if _wave_path else "見つからない（カーソルは省略）")

### コマ並べ（まず静止画で全体を把握する）

再生より軽いので、最初にこれで「いつ何が起きているか」を掴む。

In [ ]:
# ===== ここを編集 =====
MODE = "PS"        # "PS"（P/S 分離）か "T"（3成分）
ZOOM = False       # True で溝まわりの拡大
SATURATE = 0.15    # 色の頭打ち。小さくすると弱い散乱波が見える
N_TILES = 8        # 並べるコマ数
# ======================

ps = panels(MODE, ZOOM)
ext = _extent(ZOOM)
idx = np.linspace(0, ps[0][1].shape[0] - 1, N_TILES).astype(int)

fig, axes = plt.subplots(len(ps), N_TILES, figsize=(2.0 * N_TILES, 2.6 * len(ps)),
                         squeeze=False)
for r, (title, arr) in enumerate(ps):
    v = _scale(arr, SATURATE)
    for c, i in enumerate(idx):
        ax = axes[r][c]
        ax.imshow(arr[i], cmap="seismic", vmin=-v, vmax=v, extent=ext, aspect="auto")
        if not ZOOM:
            # contour は origin を指定しないと extent を無視し、索引座標のまま
            # 上下反転で描かれる。imshow と同じ origin="upper" を渡す。
            ax.contour(FIELD["mask"], levels=[0.5], colors="k", linewidths=0.6,
                       extent=ext, origin="upper")
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title("{:.2f} us".format(FIELD["steps"][i] * float(FIELD["dt"]) * 1e6),
                         fontsize=9)
        if c == 0:
            ax.set_ylabel(title, fontsize=9)
fig.suptitle("応力場  pitch{} width{} depth{} [mm]".format(
    ANI_PITCH_MM, ANI_WIDTH_MM, ANI_DEPTH_MM))
fig.tight_layout()
plt.show()

### インライン再生

ノートブック内で再生する。**全フレームを画像として HTML に埋め込むので重い。**
`FRAME_STRIDE` を上げると軽くなる。下段に A スキャン波形と時刻カーソルが出るので、
場のどの現象がどのエコーに対応するかを対応付けられる。

In [ ]:
# ===== ここを編集 =====
FRAME_STRIDE = 2   # 1 で全フレーム。重いときは 2〜4 に上げる
FPS = 10
# ======================

ps = panels(MODE, ZOOM)
ext = _extent(ZOOM)
frames = range(0, ps[0][1].shape[0], FRAME_STRIDE)
n_show = len(list(frames))
print("再生フレーム数:", n_show, "（多いと HTML が重い）")

_has_wave = WAVE is not None
fig = plt.figure(figsize=(6.0 * len(ps), 4.6 + (1.6 if _has_wave else 0)))
gs = GridSpec(2 if _has_wave else 1, len(ps), figure=fig,
              height_ratios=[3, 1] if _has_wave else [1])

ims = []
for k, (title, arr) in enumerate(ps):
    ax = fig.add_subplot(gs[0, k])
    v = _scale(arr, SATURATE)
    im = ax.imshow(arr[0], cmap="seismic", vmin=-v, vmax=v, extent=ext,
                   aspect="auto", animated=True)
    if not ZOOM:
        # contour は origin を指定しないと extent を無視し、索引座標のまま
        # 上下反転で描かれる。imshow と同じ origin="upper" を渡す。
        ax.contour(FIELD["mask"], levels=[0.5], colors="k", linewidths=0.6,
                   extent=ext, origin="upper")
    ax.set_title(title)
    ax.set_xlabel("z [mm]"); ax.set_ylabel("x [mm]")
    ims.append((im, arr))

if _has_wave:
    axw = fig.add_subplot(gs[1, :])
    t_us = np.arange(len(WAVE)) * float(FIELD["dt"]) * 1e6
    axw.plot(t_us, WAVE, lw=0.7, color="0.3")
    axw.set_xlabel("時間 [us]"); axw.set_ylabel("T1")
    axw.grid(alpha=0.3)
    cursor = axw.axvline(0, color="crimson", lw=1.5)

ttl = fig.suptitle("")


def _update(i):
    for im, arr in ims:
        im.set_array(arr[i])
    t_us_now = FIELD["steps"][i] * float(FIELD["dt"]) * 1e6
    ttl.set_text("t = {:.3f} us   (frame {}/{})".format(t_us_now, i, ims[0][1].shape[0] - 1))
    if _has_wave:
        cursor.set_xdata([t_us_now, t_us_now])
    return [im for im, _ in ims]


anim = animation.FuncAnimation(fig, _update, frames=list(frames), interval=1000 / FPS, blit=False)
plt.close(fig)

from IPython.display import HTML
HTML(anim.to_jshtml(fps=FPS))

### 動画として保存

`ffmpeg` があれば MP4、無ければ GIF（Pillow）で保存する。
出力は `tmp_output/field/` なので gitignore 対象。

In [ ]:
out_stem = REPO_ROOT / "tmp_output" / "field" / "{}_{}_p{:g}_w{:g}_d{:g}".format(
    NPZ_PATH.stem.replace("_data", ""), MODE, ANI_PITCH_MM, ANI_WIDTH_MM, ANI_DEPTH_MM)

try:
    out = out_stem.with_suffix(".mp4")
    anim.save(str(out), writer=animation.FFMpegWriter(fps=FPS, bitrate=2400))
except Exception as e:
    print("MP4 に失敗（{}）。GIF で保存する。".format(type(e).__name__))
    out = out_stem.with_suffix(".gif")
    anim.save(str(out), writer=animation.PillowWriter(fps=FPS))

print("保存:", out.relative_to(REPO_ROOT), "({:.1f} MB)".format(out.stat().st_size / 1e6))

---
## メモ

**出力先が3か所ある。** `run_simulation()` は `tmp_output/sim/` に書くが、
`calculate_avg_waveforms.py` は `tmp_output/sim_csv/` を読み、`loop_UT_sim.py` は
`data/simdata_out/` も見る。このノートブックの既存チェックは3か所すべてを見るので
無駄な再計算は起きないが、**新しい出力は `tmp_output/sim/` に入る**ので、
比較プロットに使うときはコピーか移動が要る。

**引数順に注意。** `run_simulation(f_width, f_pitch, f_depth)` は width が先頭で、
ファイル名の pitch → width → depth 順と違う。`run_one()` はキーワード引数で渡している。

**出力は gitignore 対象。** `tmp_output/` は再生成できるのでコミットしない（AGENTS.md 10章）。

**長い掃引はレポート締め切り直前に始めない。** 19条件で約50分。